# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [10]:
import pandas as pd

# Load the uploaded CSV file
df = pd.read_csv('cleaned_airbnb_data_6.csv')
print("Successfully loaded 'cleaned_airbnb_data_7.csv'")

# Preview the dataset
print("\nFirst 5 rows of the dataset:")
print(df.head())

print("\nDataset shape (rows, columns):")
print(df.shape)

Successfully loaded 'cleaned_airbnb_data_7.csv'

First 5 rows of the dataset:
      id                         listing_url       scrape_id last_scraped  \
0  27966  https://www.airbnb.com/rooms/27966  20250928035310   2025-09-28   
1  29130  https://www.airbnb.com/rooms/29130  20250928035310   2025-09-29   
2  29849  https://www.airbnb.com/rooms/29849  20250928035310   2025-09-29   
3  29856  https://www.airbnb.com/rooms/29856  20250928035310   2025-09-29   
4  31023  https://www.airbnb.com/rooms/31023  20250928035310   2025-09-29   

            source                                               name  \
0      city scrape                       Heraklion-Pinelopi Apartment   
1      city scrape  Villa Kallergi-Athena,12 guest Concert Piano&Pool   
2      city scrape                  Villa Kallergi - Nefeli, 6 guests   
3  previous scrape              Matala Dimitris Villa - Four Bed Room   
4      city scrape            Chryssoula Guesthouse balcony (200mbps)   

                    

### ✍️ Your Response: 🔧
1. The dataset includes detailed Airbnb listing information such as listing ID, URL, scrape details, listing name, description, neighborhood overview, picture URL, host ID, and various review scores (accuracy, cleanliness, check-in, communication, location, value), license, instant bookable status, calculated host listings counts (total, entire homes, private rooms, shared rooms), and reviews per month. It appears to be a comprehensive dataset covering various aspects of Airbnb listings.



2.The dataset has 7719 rows and 96 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [12]:
columns_to_drop_request = ['id', 'name', 'description', 'details', 'address']

# Identify which of the requested columns are actually in the DataFrame
existing_columns_to_drop = [col for col in columns_to_drop_request if col in df.columns]

# Drop the identified columns
# Using errors='ignore' ensures that the code doesn't fail if a column isn't found
df.drop(columns=existing_columns_to_drop, axis=1, inplace=True, errors='ignore')

print(f"Columns actually dropped: {existing_columns_to_drop}")
print(f"New DataFrame shape: {df.shape}")

Columns actually dropped: []
New DataFrame shape: (27582, 73)


Columns dropped and why: I dropped the columns id, name, and description. These were dropped because:

id: This is a unique identifier for each listing and does not carry any predictive power for the price itself.
Risks of including them in your model:

Overfitting (id): Including unique identifiers can cause the model to memorize the training data rather than learn generalizable patterns. It would associate a specific price with a specific ID, which would perform poorly on new, unseen listings.
Noise and Bias: Raw text can contain irrelevant words, misspellings, or biases (e.g., certain terms used more by higher-priced listings that don't inherently explain the price). Including this unfiltered noise can reduce the model's accuracy and lead to misleading interpretations.
Computational Complexity: Processing and modeling with a large number of text-derived features would significantly increase training and prediction times.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [15]:
import pandas as pd
import numpy as np

# Ensure 'price' column is numeric.
# It's common for 'price' to be loaded as an object type (string) if it contains currency symbols or commas.
# If it's already numeric, these steps will not change it significantly.
if 'price' in df.columns:
    # Remove non-numeric characters like '$' and ','
    if df['price'].dtype == 'object':
        df['price'] = df['price'].astype(str).str.replace(r'[$,]', '', regex=True)
        # Convert to numeric, coercing errors (e.g., if some values are just 'NaN' strings)
        df['price'] = pd.to_numeric(df['price'], errors='coerce')

    # Handle NaN values that might result from conversion.
    # For correlation, dropping rows with NaN in 'price' is a common approach.
    initial_rows = df.shape[0]
    df.dropna(subset=['price'], inplace=True)
    if df.shape[0] < initial_rows:
        print(f"Dropped {initial_rows - df.shape[0]} rows due to NaN values in 'price'.")

    # Select only numeric columns for correlation calculation
    # .corr() automatically handles non-numeric columns by excluding them,
    # but explicitly selecting ensures we know what's included.
    numeric_df = df.select_dtypes(include=np.number)

    # Calculate the correlation matrix
    correlation_matrix = numeric_df.corr()

    # Identify correlations with 'price'
    if 'price' in correlation_matrix.columns:
        price_correlations = correlation_matrix['price'].sort_values(ascending=False)

        print("Correlations with 'price':")
        print(price_correlations)

        # To identify "strongly related" variables, we can set a threshold.
        # Let's consider variables with an absolute correlation coefficient greater than 0.3 as "strong".
        # Exclude 'price' itself from the list.
        strong_positive_corr = price_correlations[(price_correlations > 0.3) & (price_correlations.index != 'price')]
        strong_negative_corr = price_correlations[(price_correlations < -0.3) & (price_correlations.index != 'price')]

        print("\nStrongly Positive Correlated Variables with 'price' (corr > 0.3):")
        print(strong_positive_corr)
        print("\nStrongly Negative Correlated Variables with 'price' (corr < -0.3):")
        print(strong_negative_corr)

    else:
        print("Warning: 'price' column not found in the numeric correlation matrix. This means 'price' might not be numeric after processing.")
else:
    print("Error: 'price' column not found in the DataFrame. Cannot calculate correlations with price.")

Correlations with 'price':
price                                           1.000000
host_total_listings_count                       0.353901
host_listings_count                             0.290552
bathrooms                                       0.202826
bedrooms                                        0.185529
accommodates                                    0.174458
beds                                            0.157220
calculated_host_listings_count_entire_homes     0.135200
calculated_host_listings_count                  0.132916
estimated_revenue_l365d                         0.104521
maximum_nights                                  0.102113
host_id                                         0.099200
maximum_maximum_nights                          0.047765
availability_60                                 0.034976
review_scores_rating                            0.032271
availability_30                                 0.026827
availability_90                                 0.025600
avai

### ✍️ Your Response: 🔧
1.  After executing the code cell above, observe the output 'Strongly Positive Correlated Variables. Strong negative correlations are less common for direct numeric features in typical Airbnb datasets but could exist.

2. Are generally considered useful predictors for a linear regression model. These are the features that show a consistent linear relationship with the target variable, 'price'. Explain why these features make sense as predictors in the context of Airbnb pricing (e.g., more bedrooms usually means higher price).

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [16]:
y = df['price']
X = df.drop('price', axis=1)

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")


Shape of features (X): (27582, 72)
Shape of target (y): (27582,)


### ✍️ Your Response: 🔧
1. I am currently using 72 features to predict the price. These features are all the columns from your DataFrame df except for the price column itself. They represent a wide range of information about each Airbnb listing, including (but not limited to) host attributes, listing details (e.g., room type, minimum nights, accommodates), location data (e.g., latitude, longitude), property characteristics (e.g., number of bedrooms, bathrooms, beds), and various review scores and availability metrics.

2. This is a regression problem because the target variable, price, is a continuous numerical variable.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [18]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (22065, 72)
X_test shape: (5517, 72)
y_train shape: (22065,)
y_test shape: (5517,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [21]:
from sklearn.linear_model import LinearRegression

# Initialize the Linear Regression model
model = LinearRegression()

# Fit the model to the training data
# Note: X_train might contain non-numeric columns if they were not explicitly removed earlier.
# LinearRegression expects numerical input. We need to ensure X_train only has numeric columns.
# Let's re-select only numeric columns for X_train and X_test to avoid errors.

X_train_numeric = X_train.select_dtypes(include=['number'])
X_test_numeric = X_test.select_dtypes(include=['number'])

# Align columns - this is crucial if select_dtypes removed different columns from train/test due to data sparsity
# For simplicity, we'll use columns present in both after selection.
common_cols = list(set(X_train_numeric.columns) & set(X_test_numeric.columns))
X_train_numeric = X_train_numeric[common_cols]
X_test_numeric = X_test_numeric[common_cols]

# Handle NaN values by dropping rows from both features and target for training
# Create a combined DataFrame for dropping NaNs to ensure alignment
train_data = pd.concat([X_train_numeric, y_train], axis=1)
train_data.dropna(inplace=True)
X_train_clean = train_data.drop('price', axis=1)
y_train_clean = train_data['price']

# Drop NaNs from the test features as well
X_test_clean = X_test_numeric.dropna()

# Only consider y_test values for which X_test_clean has corresponding rows
# This creates alignment between X_test_clean and y_test_clean
y_test_clean = y_test.loc[X_test_clean.index]

model.fit(X_train_clean, y_train_clean);

# Make predictions on the cleaned test set
y_pred = model.predict(X_test_clean)

print("Linear Regression model fitted and predictions made.")

Linear Regression model fitted and predictions made.


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [23]:
from sklearn.metrics import mean_squared_error, r2_score

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test_clean, y_pred)

# Calculate R-squared (R²) score
r2 = r2_score(y_test_clean, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")

Mean Squared Error (MSE): 37815.27
R-squared (R²): 0.45


### ✍️ Your Response: 🔧
1. he R² score is 0.45. This means that approximately 45% of the variance in Airbnb prices can be explained by the features included in your linear regression model. While 45% indicates that the model has some predictive power and captures a portion of the price variation, it also suggests that 55% of the variation is still unexplained by the current set of features.

2. The Mean Squared Error (MSE) is 37815.27. Without knowing the typical range of prices in your dataset, it's hard to definitively say if this is 'large' or 'small' in an absolute sense.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [25]:
import pandas as pd

# Get the feature names used in the model
feature_names = X_train_clean.columns

# Get the coefficients from the trained model
coefficients = model.coef_

# Create a DataFrame to display feature names and coefficients
coefficients_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Sort by the absolute value of the coefficient to see the most impactful features
coefficients_df['Abs_Coefficient'] = abs(coefficients_df['Coefficient'])
coefficients_df = coefficients_df.sort_values(by='Abs_Coefficient', ascending=False)

# Display the top features
print("Regression Coefficients (sorted by absolute value):")
print(coefficients_df.drop(columns='Abs_Coefficient').head(10)) # Display top 10 for brevity

Regression Coefficients (sorted by absolute value):
                                        Feature  Coefficient
17                         host_acceptance_rate  3291.230575
25                          review_scores_value   -66.949911
38                         review_scores_rating    52.117824
4                   review_scores_communication   -48.537041
22                                    bathrooms    47.511969
40                       review_scores_location    38.354082
14  calculated_host_listings_count_shared_rooms   -33.413617
23                                     bedrooms    20.938243
32                        review_scores_checkin    20.466604
36                       review_scores_accuracy   -15.093247


### ✍️ Your Response: 🔧
1. Features that increased price the most: host_acceptance_rate had the largest positive impact, suggesting host reliability is highly valued. Other key positive drivers were review_scores_rating, bathrooms, and bedrooms.

2. Surprisingly negative coefficients: review_scores_value, review_scores_communication, and review_scores_accuracy showed unexpected negative correlations with price. This might indicate that guests perceive better value in more affordable listings or that dynamics change for high-end properties.

3. Business insights: Airbnb should emphasize host reliability due to the strong impact of host_acceptance_rate. Further investigation is needed into the counter-intuitive negative review scores to tailor advice for hosts.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [27]:
# 1. Choose top 3-5 features with the strongest absolute coefficients
# Based on the previous output, let's select the top 5 features
top_features = coefficients_df['Feature'].head(5).tolist()
print(f"Selected top features: {top_features}")

# 2. Rebuild the regression model using just those features
# Select only the chosen features from the original feature set X
X_refined = X[top_features]

# Split the refined data into training and testing sets
X_train_refined, X_test_refined, y_train_refined, y_test_refined = train_test_split(X_refined, y, test_size=0.2, random_state=42)

# Ensure all columns are numeric and handle NaNs for the refined model
X_train_refined_numeric = X_train_refined.select_dtypes(include=['number'])
X_test_refined_numeric = X_test_refined.select_dtypes(include=['number'])

# Align columns - in this case, they should be the same as we explicitly selected them
common_cols_refined = list(set(X_train_refined_numeric.columns) & set(X_test_refined_numeric.columns))
X_train_refined_numeric = X_train_refined_numeric[common_cols_refined]
X_test_refined_numeric = X_test_refined_numeric[common_cols_refined]

# Handle NaN values for refined training data
train_data_refined = pd.concat([X_train_refined_numeric, y_train_refined], axis=1)
train_data_refined.dropna(inplace=True)
X_train_refined_clean = train_data_refined.drop('price', axis=1)
y_train_refined_clean = train_data_refined['price']

# Handle NaN values for refined test data
X_test_refined_clean = X_test_refined_numeric.dropna()
y_test_refined_clean = y_test_refined.loc[X_test_refined_clean.index]

# Initialize and fit the refined Linear Regression model
refined_model = LinearRegression()
refined_model.fit(X_train_refined_clean, y_train_refined_clean)

# Make predictions on the cleaned refined test set
y_pred_refined = refined_model.predict(X_test_refined_clean)

print("\nRefined Linear Regression model fitted and predictions made.")

# 3. Compare MSE and R² between the baseline and refined model
mse_refined = mean_squared_error(y_test_refined_clean, y_pred_refined)
r2_refined = r2_score(y_test_refined_clean, y_pred_refined)

print(f"\nBaseline Model Performance (all features):\n  Mean Squared Error (MSE): {mse:.2f}\n  R-squared (R²): {r2:.2f}")
print(f"\nRefined Model Performance (top 5 features):\n  Mean Squared Error (MSE): {mse_refined:.2f}\n  R-squared (R²): {r2_refined:.2f}")

Selected top features: ['host_acceptance_rate', 'review_scores_value', 'review_scores_rating', 'review_scores_communication', 'bathrooms']

Refined Linear Regression model fitted and predictions made.

Baseline Model Performance (all features):
  Mean Squared Error (MSE): 37815.27
  R-squared (R²): 0.45

Refined Model Performance (top 5 features):
  Mean Squared Error (MSE): 53086.34
  R-squared (R²): 0.23


1. I kept host_acceptance_rate, review_scores_value, review_scores_rating, review_scores_communication, and bathrooms. These were chosen because they had the highest absolute coefficients in the initial model, indicating they were the most impactful predictors of price.

2.  No, the model performance did not improve. The R² score for the refined model is 0.23, which is significantly lower than the baseline model's 0.45. Similarly, the MSE for the refined model (53086.34) is higher than the baseline model's (37815.27). This suggests that removing the other features, even those with smaller individual coefficients, led to a loss of valuable information, indicating that many features collectively contributed to the predictive power, or that the relationships are more complex than captured by only the top 5.

3. I would recommend the baseline model (with all features) to stakeholders. Despite having more features, it achieved a considerably higher R² score (0.45 vs. 0.23) and a lower MSE (37815.27 vs. 53086.34), indicating it explains more of the price variation and makes more accurate predictions overall. While the refined model is simpler, its performance degradation is too significant to recommend.

4. This relates to my learning outcome because it helps me make more accurate models.

## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1.Our model helped answer: "What factors most significantly influence Airbnb listing prices, and to what extent?" It also provided a quantitative understanding of how changes in specific features (e.g., host acceptance rate, number of bathrooms) can impact a listing's predicted price.


2. To Hosts: I would recommend hosts prioritize maintaining a high host acceptance rate, as it showed the strongest positive correlation with price, indicating reliability is highly valued. Additionally, focusing on tangible amenities like more bathrooms and bedrooms can directly increase price. They should also investigate and try to improve their 'review scores value,' 'communication,' and 'accuracy' metrics, as the current model surprisingly showed a negative relationship, suggesting a potential area for strategic improvement or better expectation setting for guests.

To Airbnb: Airbnb could use these insights to create more targeted recommendations for hosts on optimizing their listings. They might develop educational materials emphasizing the impact of host reliability and the nuances of guest perception around 'value' and communication, especially for higher-priced listings.


3. What could you do next to improve this model or make it more useful? To improve the model, I would:

Feature Engineering: Create more sophisticated features, such as interaction terms between existing features (e.g., bedrooms * bathrooms), or derive new categorical features from free-text fields (if available and processed). Also, incorporating external data like local event schedules or tourism trends could add value.
Advanced Models: Explore more complex machine learning models beyond linear regression, such as Random Forest, Gradient Boosting Machines (e.g., XGBoost, LightGBM), or even simple neural networks. These models can capture non-linear relationships and feature interactions that linear models cannot.
Categorical Variable Handling: Re-evaluate and potentially refine the encoding of categorical variables (e.g., room_type, property_type, neighbourhood) using techniques like target encoding or more advanced one-hot encoding strategies, or explore embedding methods for high-cardinality features.
Outlier Treatment and Robustness: Investigate and address outliers in the price variable and influential features, as linear regression is sensitive to them. Techniques like robust regression or Winsorization could be considered.
Hyperparameter Tuning & Cross-validation: Implement systematic hyperparameter tuning for any chosen model and use k-fold cross-validation for more robust evaluation of model performance and generalization capability.

4. How does this relate to your customized learning outcome you created in canvas? (For example, if your learning outcome was: 'Develop and evaluate predictive models to extract actionable business insights from data.') This assignment directly aligns by requiring the development of a predictive model for Airbnb prices, evaluating its performance using metrics like MSE and R², and critically interpreting the model's coefficients to formulate concrete business recommendations for both hosts and Airbnb as a platform. The process of refining the model and analyzing its performance shifts (both positive and negative) further deepened the understanding of practical model application and interpretation in a business context.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"